# 01 — Preprocesamiento
Genera `datos_procesados/` con ULB y bank.xlsx.
Corre todas las celdas de arriba a abajo. Necesitas `creditcard.csv` y `bank.xlsx` en esta carpeta.

In [ ]:
import os, datetime as _dt

def iniciar_ejecucion(nombre, config=None, base="resultados"):
    os.makedirs(base, exist_ok=True)
    nums = []
    for d in os.listdir(base):
        if d.startswith("ejecucion_") and os.path.isdir(os.path.join(base, d)):
            try: nums.append(int(d.split("_")[1]))
            except: pass
    n = max(nums)+1 if nums else 1
    carpeta = os.path.join(base, f"ejecucion_{n}")
    os.makedirs(carpeta)
    with open(os.path.join(carpeta, "config.txt"), "w", encoding="utf-8") as f:
        f.write("notebook : " + str(nombre) + "\n")
        f.write("fecha    : " + _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n")
        if config:
            for k, v in config.items():
                f.write(str(k) + " : " + str(v) + "\n")
    print("[Ejecucion " + str(n) + "] Resultados en: " + carpeta)
    return carpeta

def log(texto, carpeta):
    print(texto)
    with open(os.path.join(carpeta, "resultados.txt"), "a", encoding="utf-8") as f:
        f.write(str(texto) + "\n")

def guardar_csv(resultados, ruta):
    if not resultados: return
    cols = list(resultados[0].keys())
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(",".join(cols) + "\n")
        for r in resultados:
            f.write(",".join(str(r[c]) for c in cols) + "\n")


In [ ]:
import numpy as np, pandas as pd, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
SEED=42; np.random.seed(SEED)
RUTA_ULB="creditcard.csv"; RUTA_BANK="bank.xlsx"; DIR="datos_procesados"; PROP_FRAUDE=0.02
carpeta=iniciar_ejecucion("01_preprocesamiento",{"ulb":RUTA_ULB,"bank":RUTA_BANK,"prop":PROP_FRAUDE})
os.makedirs(DIR,exist_ok=True)

In [ ]:
log("=== ULB ===",carpeta)
df=pd.read_csv(RUTA_ULB); n0=len(df)
df=df.drop_duplicates().reset_index(drop=True)
log(f"Duplicados eliminados: {n0-len(df)}",carpeta)
y=df["Class"].values; X=df.drop(columns=["Class"])
Xtr,Xte,ytr,yte=train_test_split(X,y,train_size=0.8,stratify=y,random_state=SEED)
cv=[c for c in X.columns if c.startswith("V")]
ev,eva,et=StandardScaler(),RobustScaler(),StandardScaler()
def tf_(Xd,fit=False):
    Xd=Xd.copy()
    if fit:
        Xd[cv]=ev.fit_transform(Xd[cv]); Xd["Amount"]=eva.fit_transform(Xd[["Amount"]]); Xd["Time"]=et.fit_transform(Xd[["Time"]])
    else:
        Xd[cv]=ev.transform(Xd[cv]); Xd["Amount"]=eva.transform(Xd[["Amount"]]); Xd["Time"]=et.transform(Xd[["Time"]])
    return Xd.values.astype(np.float32)
Xtr=tf_(Xtr,True); Xte=tf_(Xte)
log(f"Train {Xtr.shape} fraudes={int(ytr.sum())} | Test {Xte.shape} fraudes={int(yte.sum())}",carpeta)
np.savez_compressed(os.path.join(DIR,"ulb.npz"),X_train=Xtr,X_test=Xte,y_train=ytr,y_test=yte)

In [ ]:
log("=== bank.xlsx ===",carpeta)
b=pd.read_excel(RUTA_BANK); b.columns=[c.strip() for c in b.columns]
b=b.drop(columns=["."],errors="ignore")
b["Account No"]=b["Account No"].astype(str).str.replace("'","",regex=False)
b=b.drop(columns=["CHQ.NO."],errors="ignore")
for c in ["WITHDRAWAL AMT","DEPOSIT AMT"]:
    b[c]=pd.to_numeric(b[c],errors="coerce").fillna(0.0)
b["BALANCE AMT"]=pd.to_numeric(b["BALANCE AMT"],errors="coerce"); b=b.dropna(subset=["BALANCE AMT"])
b["DATE"]=pd.to_datetime(b["DATE"],errors="coerce")
b=b.dropna(subset=["DATE"]).sort_values(["Account No","DATE"]).reset_index(drop=True)
b["MONTO"]=b["DEPOSIT AMT"]-b["WITHDRAWAL AMT"]; b["ES_RETIRO"]=(b["WITHDRAWAL AMT"]>0).astype(int)
g=b.groupby("Account No")
b["DIAS_DESDE_ANT"]=g["DATE"].diff().dt.days.fillna(0)
b["MONTO_PROM_5"]=g["MONTO"].rolling(5,min_periods=1).mean().reset_index(level=0,drop=True)
b["DESV_MONTO"]=(b["MONTO"]-b["MONTO_PROM_5"]).abs()
FEAT=["WITHDRAWAL AMT","DEPOSIT AMT","BALANCE AMT","MONTO","ES_RETIRO","DIAS_DESDE_ANT","MONTO_PROM_5","DESV_MONTO"]
esc=StandardScaler(); Xn=esc.fit_transform(b[FEAT]).astype(np.float32)
np.savez_compressed(os.path.join(DIR,"bank_normal.npz"),X_normal=Xn,features=np.array(FEAT))
log(f"Corpus normal: {Xn.shape}",carpeta)

In [ ]:
b2=b.copy(); b2["FRAUD"]=0; nf=int(len(b2)*PROP_FRAUDE)
rng=np.random.default_rng(SEED); cuentas=b2["Account No"].unique()
p99=b2.groupby("Account No")["WITHDRAWAL AMT"].quantile(0.99); fr=[]
for i in range(nf):
    cta=rng.choice(cuentas)
    base=b2[b2["Account No"]==cta].sample(1,random_state=int(rng.integers(1e6))).iloc[0].copy()
    p=i%3
    if p==0: base["WITHDRAWAL AMT"]=p99[cta]*rng.uniform(3,10); base["DEPOSIT AMT"]=0.0
    elif p==1: base["DIAS_DESDE_ANT"]=0; base["WITHDRAWAL AMT"]=max(base["WITHDRAWAL AMT"],1.0)*rng.uniform(2,5)
    else: base["WITHDRAWAL AMT"]=abs(base["BALANCE AMT"])*rng.uniform(1.5,3); base["DEPOSIT AMT"]=0.0
    base["MONTO"]=base["DEPOSIT AMT"]-base["WITHDRAWAL AMT"]; base["ES_RETIRO"]=1
    base["DESV_MONTO"]=abs(base["MONTO"]-base["MONTO_PROM_5"]); base["FRAUD"]=1; fr.append(base)
ds=pd.concat([b2,pd.DataFrame(fr)],ignore_index=True)
yb=ds["FRAUD"].values; Xb=ds[FEAT].values
Xtr2,Xte2,ytr2,yte2=train_test_split(Xb,yb,train_size=0.8,stratify=yb,random_state=SEED)
e2=StandardScaler().fit(Xtr2)
np.savez_compressed(os.path.join(DIR,"bank_sintetico.npz"),
    X_train=e2.transform(Xtr2).astype(np.float32),X_test=e2.transform(Xte2).astype(np.float32),
    y_train=ytr2,y_test=yte2,features=np.array(FEAT))
log(f"Fraude sintetico: {nf} | Train {Xtr2.shape} fraudes={int(ytr2.sum())} | Test {Xte2.shape} fraudes={int(yte2.sum())}",carpeta)
log("\nListo. Datos en ./"+DIR,carpeta)